# Qwen2.5-1.5B-Instruct QLoRA Fine-Tune on nhankins/legal_contracts

This notebook fine-tunes a legal QA model using the CUAD-derived dataset on Hugging Face.
Target: `Qwen/Qwen2.5-1.5B-Instruct` on a T4 (16GB) with QLoRA.


In [1]:
# Install dependencies
!pip -q install -U transformers datasets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.8 MB/s eta 0:00:00


In [2]:
# Check GPU
!nvidia-smi

Tue Mar 10 16:48:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Optional: Hugging Face token to avoid rate limits
import os
# os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
output_dir = 'qwen2.5-1.5b-legal-qlora'
max_len = 1024  # T4-friendly
num_train_epochs = 3
train_limit = None  # set to int for quick runs (e.g., 2000)


In [4]:
# Reuse fine-tuned weights if available
from pathlib import Path
reuse_choice = input("Reuse existing fine-tuned weights if found? [y/N]: ").strip().lower()
reuse_if_available = reuse_choice in {"y", "yes"}
output_path = Path(output_dir)
weight_files = [
    "adapter_model.safetensors", "adapter_model.bin", "adapter_config.json",
    "pytorch_model.bin", "model.safetensors"
]
weights_exist = any((output_path / f).exists() for f in weight_files)
SKIP_TRAINING = bool(reuse_if_available and weights_exist)
if SKIP_TRAINING:
    print(f"Found weights in {output_dir}. Training will be skipped.")
elif reuse_if_available:
    print("No saved weights found. Training will run.")
else:
    print("Training will run.")


Reuse existing fine-tuned weights if found? [y/N]: N
Training will run.


In [5]:
# Build QA prompts from a script-free CUAD dataset (works with datasets>=4.x)
if SKIP_TRAINING:
    print("Skipping dataset prep (reusing weights).")
else:
    from datasets import load_dataset

    ds = load_dataset("umarbutler/better-cuad")  # parquet-backed
    raw = ds["train"]

    if train_limit:
        raw = raw.select(range(min(train_limit, len(raw))))

    answer_cols = [c for c in raw.column_names if c.endswith("-Answer")]
    text_col = "Text"
    max_questions_per_doc = 8  # keep training size sane on T4

    def build_batch(batch):
        texts = []
        for i in range(len(batch[text_col])):
            text = batch[text_col][i]
            if not text:
                continue
            count = 0
            for col in answer_cols:
                ans = batch[col][i]
                if ans is None:
                    continue
                ans = str(ans).strip()
                if not ans or ans.lower() in {"n/a", "na", "none", "no"}:
                    continue
                clause = col[:-7]  # remove "-Answer"
                prompt = (
                    "You are a legal contract assistant. Answer using only the provided context.\n\n"
                    f"Question: Highlight the parts (if any) of this contract related to {clause}.\n\n"
                    f"Context:\n{text}\n\n"
                    f"Answer:\n{ans}"
                )
                texts.append(prompt)
                count += 1
                if count >= max_questions_per_doc:
                    break
        return {"text": texts}

    qa = raw.map(build_batch, batched=True, remove_columns=raw.column_names)
    qa = qa.shuffle(seed=42)
    split = qa.train_test_split(test_size=0.05, seed=42)
    train = split["train"]
    eval_ = split["test"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/812 [00:00<?, ?B/s]

cuad.jsonl:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/510 [00:00<?, ? examples/s]

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

In [6]:
# Tokenize + load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_len)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

if SKIP_TRAINING:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, output_dir)
else:
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lora_config)

    train_tok = train.map(tokenize, batched=True, remove_columns=["text"])
    eval_tok = eval_.map(tokenize, batched=True, remove_columns=["text"])


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Map:   0%|          | 0/3582 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

In [7]:
# Train (compatible across transformers versions)
if SKIP_TRAINING:
    print(f"Skipping training. Using weights from {output_dir}")
else:
    from transformers import TrainingArguments

    train_args_kwargs = dict(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=num_train_epochs,
        fp16=True,
        logging_steps=50,
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    )

    # Handle version differences
    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        train_args_kwargs["eval_strategy"] = "steps"
    else:
        train_args_kwargs["evaluation_strategy"] = "steps"

    args = TrainingArguments(**train_args_kwargs)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=data_collator,
    )

    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,1.607906,1.523154
100,1.443721,1.358387
150,1.269695,1.228841
200,1.136593,1.077766
250,0.900448,0.918862
300,0.700596,0.737145
350,0.549956,0.566352
400,0.425565,0.432233
450,0.328128,0.342070
500,0.197069,0.265349


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss,Validation Loss
50,1.607906,1.523154
100,1.443721,1.358387
150,1.269695,1.228841
200,1.136593,1.077766
250,0.900448,0.918862
300,0.700596,0.737145
350,0.549956,0.566352
400,0.425565,0.432233
450,0.328128,0.342070
500,0.197069,0.265349


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

In [8]:
!pip -q install pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 7.2 MB/s eta 0:00:00


In [9]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))

Saving 85 Altamar Jul 24 Contract.pdf to 85 Altamar Jul 24 Contract.pdf


In [10]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text() or ""
print("Chars:", len(text))


Chars: 145403


In [11]:
def chunk_text(t, size=1200, overlap=200):
    t = " ".join(t.split())
    chunks = []
    i = 0
    while i < len(t):
        chunks.append(t[i:i+size])
        i += size - overlap
    return chunks

chunks = chunk_text(text)
print("Chunks:", len(chunks))


Chunks: 143


In [12]:
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModel

emb_model = "BAAI/bge-base-en-v1.5"
emb_tok = AutoTokenizer.from_pretrained(emb_model)
emb_net = AutoModel.from_pretrained(emb_model).eval()

def embed(texts):
    inputs = emb_tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        out = emb_net(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
    return pooled.cpu().numpy().astype("float32")

embeddings = embed(chunks)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def retrieve(query, k=4):
    q = embed([query])
    scores, idx = index.search(q, k)
    return [chunks[i] for i in idx[0]]


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
import re

NAME_LINE_RE = re.compile(r"Name:\s*([^\n\r]+)")
PARTY_QUERY_RE = re.compile(
    r"\b(who|name|names|identify|list)\b.*\b(contract[\s-]?holders?|tenant|landlord|parties?)\b"
    r"|\b(contract[\s-]?holders?|tenant|landlord|parties?)\b.*\b(who|name|names|identify|list)\b",
    re.IGNORECASE,
)

def extract_names_from_text(text):
    lines = NAME_LINE_RE.findall(text)
    return [l.strip() for l in lines if l.strip()]

def answer_from_pdf(question, k=4):
    if PARTY_QUERY_RE.search(question):
        names = extract_names_from_text(text)
        # pick the first Name: line after the "contract-holder" label if you want more accuracy
        return "Names found: " + "; ".join(names)

    # otherwise fall back to RAG + LLM
    ctx = "\n\n".join(retrieve(question, k))
    prompt = (
        "You are a legal contract assistant. Answer using only the provided context.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{ctx}\n\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()
